# 🧬 Prédiction de l'âge à partir de la méthylation de l'ADN (horloge épigénétique)

**Objectif** : prédire l'âge chronologique d'une personne à partir des niveaux de méthylation de son ADN
(valeurs « bêta » entre 0 et 1 sur des milliers de sites **CpG**), dans la continuité de la compétition **Epiclock**.

**Le défi** : beaucoup plus de variables que d'échantillons (*p ≫ n*). Une régression classique sur-apprend parfaitement :
il faut **sélectionner** et **régulariser**.

Plan :
1. Données (fichier CSV échantillons × CpG avec une colonne `age`)
2. Transformation de l'âge de Horvath
3. Pipeline sans fuite : imputation → filtre de variance → sélection des CpG **dans chaque pli** → modèle
4. ElasticNet, Lasso, gradient boosting, ensemble ; **validation croisée imbriquée**
5. Vérification d'absence de fuite : âges mélangés au hasard

> 💡 **Sur Google Colab** : *Tout exécuter* (le CPU suffit, ~15 à 25 minutes au total, dont le téléchargement des données).

In [ ]:
# CONFIG
DATA_PATH = "data/methylation.csv"   # lignes = échantillons, colonnes = CpG + une colonne d'âge
TARGET = "age"
K_CPG = 2000                         # nombre de CpG gardés par la présélection (dans chaque pli)
N_FOLDS = 5

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import pearsonr
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import TransformedTargetRegressor
from sklearn.ensemble import HistGradientBoostingRegressor, VotingRegressor
from sklearn.feature_selection import VarianceThreshold
from sklearn.impute import SimpleImputer
from sklearn.linear_model import ElasticNetCV, LassoCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, median_absolute_error
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

### Télécharger les vraies données : GSE40279 (Hannum et al., 2013)
Jeu public : **656 échantillons de sang** d'adultes (19 à 101 ans), puce Illumina 450k (~485 000 sites CpG), sur la base GEO du NCBI.
La cellule suivante télécharge le fichier (~0,5 à 1 Go), lit l'âge de chaque personne et le tableau des valeurs bêta, puis le réduit :
- on retire les CpG avec plus de 5 % de valeurs manquantes ;
- on garde les **50 000 CpG les plus variables**.

Ce filtre est **non supervisé** : il n'utilise pas l'âge, donc il ne crée pas de fuite (la sélection selon l'âge, elle, est refaite dans
chaque pli, voir section 3). Il ramène les données de ~2,5 Go à ~250 Mo en mémoire. Le résultat est enregistré dans `data/methylation.csv`
(non versionné). Compter 5 à 10 minutes sur Colab, une seule fois.


In [ ]:
import gzip, re, urllib.request

GEO_URL = "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE40nnn/GSE40279/matrix/GSE40279_series_matrix.txt.gz"
GZ_PATH = "data/GSE40279_series_matrix.txt.gz"
N_TOP_VARIANCE = 50_000

def prepare_gse40279(gz_path, out_path, n_top=N_TOP_VARIANCE, max_missing=0.05):
    # 1) en-tête : identifiants des échantillons et âges
    sample_ids, ages, table_start = None, None, None
    with gzip.open(gz_path, "rt") as f:
        for i, line in enumerate(f):
            fields = [v.strip().strip('"') for v in line.rstrip("\n").split("\t")]
            if fields[0] == "!Sample_geo_accession":
                sample_ids = fields[1:]
            elif fields[0] == "!Sample_characteristics_ch1" and re.match(r"age", fields[1], re.I):
                ages = [float(re.search(r"([\d.]+)\s*$", v).group(1)) for v in fields[1:]]
            elif fields[0] == "!series_matrix_table_begin":
                table_start = i + 1
                break
    assert sample_ids and ages and len(ages) == len(sample_ids), "âges introuvables dans l'en-tête"

    # 2) tableau CpG × échantillons, lu par morceaux pour limiter la mémoire
    kept = []
    reader = pd.read_csv(gz_path, sep="\t", skiprows=table_start, index_col=0, comment="!",
                         na_values=["null", "NA", ""], chunksize=50_000)
    for chunk in reader:
        chunk = chunk.astype(np.float32)
        kept.append(chunk[chunk.isna().mean(axis=1) <= max_missing])
    beta = pd.concat(kept)
    print(f"{beta.shape[0]:,} CpG gardés après le filtre des valeurs manquantes")
    beta = beta.loc[beta.var(axis=1).nlargest(min(n_top, len(beta))).index]

    # 3) une ligne par échantillon, une colonne par CpG, plus l'âge
    df = beta.T
    df.index.name = "sample"
    assert list(df.index) == sample_ids
    df.insert(0, "age", ages)
    df.to_csv(out_path, float_format="%.4f")
    print(f"Enregistré : {out_path} ({df.shape[0]} échantillons × {df.shape[1] - 1:,} CpG)")

if not os.path.exists(DATA_PATH):
    os.makedirs("data", exist_ok=True)
    if not os.path.exists(GZ_PATH):
        print("Téléchargement de GSE40279 depuis GEO...")
        urllib.request.urlretrieve(GEO_URL, GZ_PATH)
    prepare_gse40279(GZ_PATH, DATA_PATH)


## 1. Les données
Si le fichier n'existe pas, on génère des **données synthétiques** (40 CpG « horloge » + du bruit) uniquement pour tester le notebook.
Pour de vraies données : celles de la compétition Epiclock, ou le jeu public **GSE40279** (Hannum et al., 2013 ; 656 échantillons de sang) —
téléchargé automatiquement par la cellule précédente.

In [ ]:
def horvath_transform(age, adult_age=20.0):
    age = np.asarray(age, dtype=float)
    return np.where(age <= adult_age, np.log(np.clip(age, -0.999, None) + 1) - np.log(adult_age + 1),
                    (age - adult_age) / (adult_age + 1))

def horvath_inverse(y, adult_age=20.0):
    y = np.asarray(y, dtype=float)
    return np.where(y <= 0, (adult_age + 1) * np.exp(y) - 1, y * (adult_age + 1) + adult_age)

def synthetic_methylation(n=300, p=3000, n_clock=40, seed=0):
    rng = np.random.default_rng(seed)
    age = rng.uniform(0, 90, n)
    logits = rng.normal(0, 1.5, size=(n, p))
    clock = rng.choice(p, n_clock, replace=False)
    logits[:, clock] += np.outer(horvath_transform(age), rng.choice([-1, 1], n_clock) * rng.uniform(0.5, 1.5, n_clock))
    beta = 1 / (1 + np.exp(-logits))
    beta[rng.random(beta.shape) < 0.01] = np.nan
    df = pd.DataFrame(beta, columns=[f"cg{i:08d}" for i in range(p)])
    df.insert(0, "age", age.round(1))
    return df

if os.path.exists(DATA_PATH):
    data = pd.read_csv(DATA_PATH, index_col=0)
    SYNTHETIC = False
else:
    print("⚠️ Fichier introuvable : données SYNTHÉTIQUES (démonstration uniquement, pas de vraie biologie).")
    data = synthetic_methylation()
    SYNTHETIC = True

y = pd.to_numeric(data.pop(TARGET)).to_numpy(dtype=float)
X = data.apply(pd.to_numeric, errors="coerce")
print(f"{X.shape[0]} échantillons × {X.shape[1]} CpG | âge de {y.min():.0f} à {y.max():.0f} ans (moyenne {y.mean():.1f})")
print(f"Valeurs manquantes : {X.isna().mean().mean():.2%}")

## 2. La transformation de Horvath
La méthylation évolue beaucoup plus vite pendant l'enfance. Horvath (2013) ajuste donc le modèle sur `F(âge)` :
logarithmique avant 20 ans, linéaire après. `F` est continue, de même pente `1/21` des deux côtés de 20 ans, strictement croissante,
donc inversible : on retransforme les prédictions avec `F⁻¹`.

In [ ]:
ages = np.linspace(0, 100, 1001)
assert np.allclose(horvath_inverse(horvath_transform(ages)), ages)   # inversible

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(ages, horvath_transform(ages))
ax.axvline(20, ls="--", c="gray", lw=1)
ax.set_xlabel("âge (années)"); ax.set_ylabel("F(âge)"); ax.set_title("Transformation de Horvath (adulte = 20 ans)")
ax.grid(alpha=0.3); plt.show()

## 3. Un pipeline sans fuite
**Erreur classique** : choisir les CpG les plus corrélés à l'âge sur *toutes* les données, puis faire une validation croisée.
Les échantillons de test ont alors participé au choix des variables, et l'erreur est faussement optimiste (Ambroise & McLachlan, 2002).

Ici, la sélection est une étape du `Pipeline` : elle est **ré-apprise sur chaque pli d'entraînement**.

In [ ]:
class CorrelationSelector(BaseEstimator, TransformerMixin):
    # Garde les k variables les plus corrélées (|Pearson|) à la cible, calculées sur les données d'entraînement uniquement
    def __init__(self, k=2000):
        self.k = k

    def fit(self, X, y):
        X, y = np.asarray(X, dtype=float), np.asarray(y, dtype=float)
        Xc, yc = X - X.mean(axis=0), y - y.mean()
        scores = np.abs(Xc.T @ yc) / (np.sqrt((Xc ** 2).sum(axis=0) * (yc ** 2).sum()) + 1e-12)
        self.support_ = np.sort(np.argsort(scores)[::-1][:min(self.k, X.shape[1])])
        return self

    def transform(self, X):
        return np.asarray(X, dtype=float)[:, self.support_]

def preprocess(k, scale=True):
    steps = [("imputation", SimpleImputer(strategy="median")),
             ("variance", VarianceThreshold(1e-4)),
             ("selection", CorrelationSelector(k))]
    return steps + ([("standardisation", StandardScaler())] if scale else [])

def with_horvath(reg):
    return TransformedTargetRegressor(regressor=reg, func=horvath_transform, inverse_func=horvath_inverse, check_inverse=False)

def build(name, k=K_CPG):
    if name == "ElasticNet":   # horloge de Horvath : L1 + L2, pénalité choisie par validation croisée interne
        return with_horvath(Pipeline(preprocess(k) + [("modèle", ElasticNetCV(l1_ratio=0.5, cv=5, max_iter=5000, n_jobs=-1))]))
    if name == "Lasso":        # horloge plus creuse (moins de CpG)
        return with_horvath(Pipeline(preprocess(k) + [("modèle", LassoCV(cv=5, max_iter=5000, n_jobs=-1))]))
    if name == "Gradient boosting":
        gbm = HistGradientBoostingRegressor(learning_rate=0.05, max_iter=500, max_leaf_nodes=15, l2_regularization=1.0,
                                            early_stopping=True, random_state=0)
        return with_horvath(Pipeline(preprocess(min(k, 500), scale=False) + [("modèle", gbm)]))
    if name == "Ensemble":
        return VotingRegressor([("en", build("ElasticNet", k)), ("gb", build("Gradient boosting", k))])

## 4. Validation croisée imbriquée et comparaison des modèles
Boucle **externe** (5 plis) : chaque échantillon reçoit une prédiction d'un modèle qui ne l'a jamais vu.
Boucle **interne** (dans `ElasticNetCV` / `LassoCV`) : choix de la pénalité sur le train du pli uniquement.

In [ ]:
def metrics(y_true, pred):
    return {"MAE (ans)": mean_absolute_error(y_true, pred), "erreur médiane": median_absolute_error(y_true, pred),
            "RMSE": np.sqrt(mean_squared_error(y_true, pred)),
            "Pearson r": pearsonr(y_true, pred)[0] if np.std(pred) > 0 else np.nan}

outer = KFold(N_FOLDS, shuffle=True, random_state=0)
preds, rows = {}, {"Référence : âge moyen": metrics(y, np.full_like(y, y.mean()))}
for name in ["ElasticNet", "Lasso", "Gradient boosting", "Ensemble"]:
    preds[name] = cross_val_predict(build(name), X, y, cv=outer)
    rows[name] = metrics(y, preds[name])
    print(f"{name:18s} MAE = {rows[name]['MAE (ans)']:.2f} ans")
pd.DataFrame(rows).T.round(3)

In [ ]:
fig, axes = plt.subplots(1, len(preds), figsize=(4 * len(preds), 4), sharex=True, sharey=True)
lim = [min(y.min(), min(p.min() for p in preds.values())) - 2, max(y.max(), max(p.max() for p in preds.values())) + 2]
for ax, (name, p) in zip(axes, preds.items()):
    ax.scatter(y, p, s=10, alpha=0.6)
    ax.plot(lim, lim, "k--", lw=1)
    ax.set_title(f"{name}\nMAE = {mean_absolute_error(y, p):.2f} ans")
    ax.set_xlabel("âge réel")
axes[0].set_ylabel("âge prédit (hors pli)")
plt.tight_layout(); plt.show()

### Combien de CpG l'horloge utilise-t-elle ?
On réentraîne l'ElasticNet sur **toutes** les données (pour l'interprétation seulement, pas pour l'évaluation) et on compte les
coefficients non nuls.

In [ ]:
final = build("ElasticNet").fit(X, y)
pipe = final.regressor_
cpg = X.columns[pipe.named_steps["variance"].get_support()][pipe.named_steps["selection"].support_]
coefs = pd.Series(pipe.named_steps["modèle"].coef_, index=cpg)
nonzero = coefs[coefs != 0]
print(f"{len(nonzero)} CpG avec un coefficient non nul sur {len(cpg)} présélectionnés "
      f"({(nonzero > 0).sum()} se méthylent avec l'âge, {(nonzero < 0).sum()} se déméthylent)")
nonzero.reindex(nonzero.abs().sort_values(ascending=False).index).head(10).round(3).to_frame("coefficient (standardisé)")


## 5. Vérification : y a-t-il une fuite ?
On **mélange les âges au hasard** : il n'y a plus rien à apprendre. Si la sélection des CpG fuyait, le modèle semblerait
quand même battre la moyenne. Avec un pipeline correct, l'erreur doit rester au niveau de la référence « âge moyen ».

In [ ]:
y_shuffled = np.random.default_rng(0).permutation(y)
pred_shuffled = cross_val_predict(Pipeline(preprocess(50) + [("modèle", ElasticNetCV(l1_ratio=0.5, cv=5, max_iter=5000))]),
                                  X, y_shuffled, cv=outer)
mae_model = mean_absolute_error(y_shuffled, pred_shuffled)
mae_mean = mean_absolute_error(y_shuffled, np.full_like(y_shuffled, y_shuffled.mean()))
print(f"Âges mélangés : MAE modèle = {mae_model:.2f} ans | MAE « âge moyen » = {mae_mean:.2f} ans")
print("✅ Pas de fuite détectée" if mae_model > 0.9 * mae_mean else "⚠️ Suspicion de fuite : le modèle bat la moyenne sur des âges aléatoires")

## Conclusion (à compléter avec tes résultats sur les vraies données)
- Quel modèle a la plus petite MAE ? L'ensemble apporte-t-il quelque chose ?
- Quel écart entre MAE et RMSE (signe d'erreurs extrêmes, souvent chez les très jeunes ou les très âgés) ?
- Combien de CpG l'ElasticNet garde-t-il avec un coefficient non nul ?

## Références
- Horvath, *DNA methylation age of human tissues and cell types*, Genome Biology, 2013.
- Hannum et al., *Genome-wide methylation profiles reveal quantitative views of human aging rates*, Molecular Cell, 2013.
- Ambroise & McLachlan, *Selection bias in gene extraction on the basis of microarray gene-expression data*, PNAS, 2002.